<a href="https://colab.research.google.com/github/ardominguezm/golden-age-semantic-reconfiguration/blob/main/paper1_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 8 — Transition + Baroque Chronology Acquisition**

Phase 7 established **84 primary-dated sonnets** across four author groups and four historiographic stages, but the dated set remained dominated by Góngora (58/84), with `N_eff = 2.394`, and the `Baroque` stage still had no primary composition chronology.

Phase 8 is a targeted scholarly-acquisition pass. It does **not** add more Góngora. It tests:
1. four Pedro Espinosa sonnets assigned by scholarship to the 1594–1596 amorous phase;
2. a small set of Quevedo sonnets with poem-specific scholarly or historical chronology (1609, 1610, 1611, 1624);
3. stricter readiness diagnostics requiring all five historiographic stages and at least two independently represented `Transition` authors.

**No semantic network is built in this notebook.**

In [ ]:
import re, shutil, subprocess, unicodedata, math
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES={
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
 'herrera_stylistics':('https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git','0de990eac908897b5e931aeb5c496170ccf35bab'),
}
ROOT=Path('/content/gasr_phase8_sources'); ROOT.mkdir(exist_ok=True)

def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst

paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']; HS=paths['herrera_stylistics']
XML_ID='{http://www.w3.org/XML/1998/namespace}id'

def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))
def body_title(root):
    for body in root.iter():
        if local(body.tag)=='body':
            for x in body.iter():
                if local(x.tag)=='title' and el_text(x): return el_text(x)
            break
    return ''
def segment_blocks(path):
    raw=Path(path).read_text(encoding='utf-8',errors='replace').replace('\r\n','\n'); out=[]
    for i,block in enumerate(re.split(r'\n\s*\n+',raw),1):
        lines=[x.strip() for x in block.splitlines() if x.strip()]
        if lines:
            txt='\n'.join(lines); out.append({'block_id':i,'n_lines':len(lines),'text':txt,'signature':norm(txt),'first_line':lines[0],'first2_signature':norm('\n'.join(lines[:2]))})
    return pd.DataFrame(out)

rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    author=fp.parent.name; txt='\n'.join(lines); bib=[el_text(x) for x in root.iter() if local(x.tag) in {'bibl','witness'} and el_text(x)]
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'title':body_title(root),'n_lines':len(lines),'text':txt,'signature':norm(txt),'first_line':lines[0],'first_line_sig':norm(lines[0]),'first2_signature':norm('\n'.join(lines[:2])),'source_bibl':' | '.join(bib[:4]),'source_file':str(fp.relative_to(N))})
n=pd.DataFrame(rows); assert len(n)==5078,len(n)
print('Pinned sources ready')
print('Navarro poems:',len(n),'| author folders:',n.author_dir.nunique())

Pinned sources ready
Navarro poems: 5078 | author folders: 53


In [ ]:
# Temporal schema and safe assignment helpers
temporal=n[['n_id','author_dir','title','source_file','first_line','source_bibl']].copy()
for c in ['composition_min','composition_max','circulation_year','sensitivity_min','sensitivity_max','composition_not_before','composition_not_after']: temporal[c]=pd.NA
for c,v in [('temporal_confidence','unassigned'),('temporal_basis',''),('temporal_source',''),('chronology_status','undated'),('circulation_basis',''),('circulation_source',''),('sensitivity_basis',''),('sensitivity_source',''),('sensitivity_status','none')]: temporal[c]=v
constraint_log=[]

def assign_primary(ids,lo,hi,confidence,basis,source):
    ids=set(ids); mask=temporal.n_id.isin(ids)
    if int(mask.sum())!=len(ids): raise ValueError(f'Missing primary IDs: {sorted(ids-set(temporal.loc[mask,"n_id"]))[:5]}')
    for idx in temporal.index[mask]:
        new=(int(lo),int(hi),str(confidence),str(basis),str(source))
        if temporal.at[idx,'chronology_status']=='undated':
            temporal.at[idx,'composition_min']=int(lo); temporal.at[idx,'composition_max']=int(hi); temporal.at[idx,'temporal_confidence']=confidence; temporal.at[idx,'temporal_basis']=basis; temporal.at[idx,'temporal_source']=source; temporal.at[idx,'chronology_status']='primary_dated'
        else:
            old=(int(temporal.at[idx,'composition_min']),int(temporal.at[idx,'composition_max']),str(temporal.at[idx,'temporal_confidence']),str(temporal.at[idx,'temporal_basis']),str(temporal.at[idx,'temporal_source']))
            if old!=new: raise ValueError(f'Contradictory primary assignment for {temporal.at[idx,"n_id"]}: {old} vs {new}')

def assign_sensitivity(ids,lo,hi,basis,source):
    ids=set(ids); mask=temporal.n_id.isin(ids)
    if int(mask.sum())!=len(ids): raise ValueError('Missing sensitivity IDs')
    temporal.loc[mask,'sensitivity_min']=int(lo); temporal.loc[mask,'sensitivity_max']=int(hi); temporal.loc[mask,'sensitivity_basis']=basis; temporal.loc[mask,'sensitivity_source']=source; temporal.loc[mask,'sensitivity_status']='sensitivity_only'

def set_circulation(ids,year,basis,source):
    ids=set(ids); mask=temporal.n_id.isin(ids)
    if int(mask.sum())!=len(ids): raise ValueError('Missing circulation IDs')
    for idx in temporal.index[mask]:
        cur=temporal.at[idx,'circulation_year']
        if pd.isna(cur) or int(year)<int(cur):
            temporal.at[idx,'circulation_year']=int(year); temporal.at[idx,'circulation_basis']=basis; temporal.at[idx,'circulation_source']=source

def add_constraint(ids,kind,year,confidence,basis,source):
    if kind not in {'not_before','not_after'}: raise ValueError(kind)
    ids=set(ids); mask=temporal.n_id.isin(ids)
    if int(mask.sum())!=len(ids): raise ValueError('Missing constraint IDs')
    col='composition_not_before' if kind=='not_before' else 'composition_not_after'
    for idx in temporal.index[mask]:
        old=temporal.at[idx,col]
        if pd.isna(old): temporal.at[idx,col]=int(year)
        elif kind=='not_before': temporal.at[idx,col]=max(int(old),int(year))
        else: temporal.at[idx,col]=min(int(old),int(year))
        constraint_log.append({'n_id':temporal.at[idx,'n_id'],'author_dir':temporal.at[idx,'author_dir'],'constraint_kind':kind,'constraint_year':int(year),'confidence':confidence,'basis':basis,'source':source})

def resolve_incipit(author,incipit):
    z=n[n.author_dir.eq(author)].copy(); key=norm(incipit); exact=z[z.first_line_sig.eq(key)]
    if len(exact)==1: return exact.iloc[0],'exact_first_line'
    pref=z[z.first_line_sig.str.startswith(key)|z.first_line_sig.map(lambda x:key.startswith(x))]
    if len(pref)==1: return pref.iloc[0],'unique_normalized_prefix'
    return None,f'unresolved_{len(exact)}eq_{len(pref)}prefix'

def resolve_any_incipit(author,variants):
    hits=[]
    for inc in variants:
        row,method=resolve_incipit(author,inc)
        if row is not None: hits.append((row.n_id,row,method,inc))
    ids=sorted(set(x[0] for x in hits))
    if len(ids)==1:
        hit=next(x for x in hits if x[0]==ids[0]); return hit[1],hit[2],hit[3]
    return None,f'unresolved_variants_{len(ids)}targets',None

# ---- Góngora Phase 4/5 chronology ----
gfile=G/'gongora_obra-poetica.xml'; groot=ET.parse(gfile).getroot(); parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'scholarly_year':ys[0] if len(ys)==1 else pd.NA,'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False); ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict(); links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy(); collisions=set(pre.g_id.value_counts()[lambda s:s>1].index); glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left'); acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False):
    conf='A' if r.method=='exact' else 'B'; basis='scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link'; assign_primary([r.n_id],r.scholarly_year,r.scholarly_year,conf,basis,'Cátedra Góngora / Carreira-Biblioteca Castro chronology')
phase4_nids=set(glink.loc[glink.accept_phase4,'n_id']); phase4_gids=set(glink.loc[glink.accept_phase4,'g_id']); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy(); first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); diag=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[]); gid=ids2[0] if len(ids2)==1 else None; score,year,status=pd.NA,pd.NA,''
    if gid:
        gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio(); year=gr.scholarly_year; status=gr.year_status
    diag.append({'n_id':r.n_id,'unique_first2_gid':gid,'unique_first2_score':score,'unique_first2_year':year,'year_status':status})
gdiag=pd.DataFrame(diag); gdiag['preaccept']=gdiag.unique_first2_gid.notna()&pd.to_numeric(gdiag.unique_first2_score,errors='coerce').ge(0.95)&gdiag.unique_first2_year.notna()&gdiag.year_status.eq('unique')&~gdiag.unique_first2_gid.isin(phase4_gids); new_collisions=set(gdiag.loc[gdiag.preaccept,'unique_first2_gid'].value_counts()[lambda s:s>1].index); gdiag['accept_phase5']=gdiag.preaccept&~gdiag.unique_first2_gid.isin(new_collisions); new_g=gdiag[gdiag.accept_phase5].copy()
for r in new_g.itertuples(index=False): assign_primary([r.n_id],r.unique_first2_year,r.unique_first2_year,'B','scholarly_chronology_year_variant_link','Cátedra Góngora; unique first-two-line signature + >=0.95 full-text similarity')

# ---- Garcilaso, Herrera layer, Boscán sensitivity ----
roman_vals={'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}
def roman_to_int(s):
    total=prev=0
    for ch in reversed(str(s).upper()):
        v=roman_vals.get(ch,0); total+=-v if v<prev else v; prev=max(prev,v)
    return total
gar=n[n.author_dir.eq('GarcilasoDeLaVega')].copy(); gar['roman_token']=gar.title.str.extract(r'^\s*-\s*([IVXLCDM]+)\s*-\s*$',expand=False); gar['title_no']=gar.roman_token.map(lambda x:roman_to_int(x) if isinstance(x,str) else pd.NA).astype('Int64'); gar['sonnet_no']=pd.to_numeric(gar.n_id.str.extract(r'_(\d+)\.xml$',expand=False),errors='coerce').astype('Int64'); assert len(gar)==38; assert int((gar.title_no==gar.sonnet_no).fillna(False).sum())==38
GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),35:(1535,1535,'A','historically_anchored_scholarly_year'),**{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items():
    z=gar[gar.sonnet_no.eq(no)]; assert len(z)==1; assign_primary([z.iloc[0].n_id],lo,hi,conf,basis,'Lapesa chronology via E. L. Rivers (CVC) + AISO/AISPI checks')
U=HS/'corpus'/'untagged_corpus'; H14=segment_blocks(U/'H.txt'); H14=H14[H14.n_lines.eq(14)].copy(); P214=segment_blocks(U/'P2.txt'); P214=P214[P214.n_lines.eq(14)].copy(); nh=n[n.author_dir.eq('FernandoDeHerrera')].copy(); Hsig=set(H14.signature); P2sig=set(P214.signature); h_ids=nh.loc[nh.signature.isin(Hsig),'n_id'].tolist(); p2_ids=nh.loc[nh.signature.isin(P2sig)&~nh.signature.isin(Hsig),'n_id'].tolist(); set_circulation(h_ids,1582,'H / Algunas obras textual layer','Hernández-Lorenzo companion corpus; Algunas obras (1582)'); set_circulation(p2_ids,1619,'P2 / Versos posthumous textual layer','Hernández-Lorenzo companion corpus; Versos (1619)'); nb=n[n.author_dir.eq('JuanBoscan')].copy(); assign_sensitivity(nb.n_id.tolist(),1526,1542,'Boscán Italianate-sonnet activity envelope; sensitivity only','Navagero-Boscán Granada encounter (1526) to Boscán death (1542)')
primary5=temporal[temporal.chronology_status.eq('primary_dated')].copy(); conf5=primary5.temporal_confidence.value_counts(); assert len(primary5)==76; assert int(conf5.get('A',0))==14; assert int(conf5.get('B',0))==62; assert len(new_g)==5; assert int((temporal.sensitivity_status=='sensitivity_only').sum())==100; assert int(temporal.circulation_year.notna().sum())==89

# ---- Phase 6 ----
for author,year in {'JuanBoscan':1542,'FernandoDeHerrera':1597,'LuisCarrilloySotomayor':1610}.items(): add_constraint(n.loc[n.author_dir.eq(author),'n_id'].tolist(),'not_after',year,'C','author_death_upper_bound','Biographical death date; hard upper-bound constraint only')
add_constraint(h_ids,'not_after',1582,'B','exact_H_textual_attestation_TAQ','Algunas obras (1582), exact normalized-text match to H companion layer')
PRIMARY_ANCHORS=[('Cervantes','Vimos en julio otra semana santa',1596,1596,'B','historical_event_sonnet_Cadiz_1596','BNE / Cervantes scholarship: Cádiz episode, 1596'),('Cervantes','Voto a Dios que me espanta esta grandeza',1598,1598,'A','historical_event_sonnet_FelipeII_tomb_1598','Cervantes Virtual: túmulo de Felipe II in Seville, 1598'),('Cervantes','El que subió por sendas nunca usadas',1597,1598,'B','Herrera_death_epitaph_interval','Cervantes identifies poem with Herrera death; Herrera died 1597')]
for author,incipit,lo,hi,conf,basis,source in PRIMARY_ANCHORS:
    row,method=resolve_incipit(author,incipit); assert row is not None,(author,incipit,method); assign_primary([row.n_id],lo,hi,conf,basis,source)
FLORES_PHASE6=[('JuanDeArguijo','Castiga el cielo a Tántalo inhumano'),('JuanDeArguijo','A quién me quejaré del crudo engaño'),('JuanDeArguijo','La horrible sima con espanto mira'),('JuanDeArguijo','Si pudo de Anfión el dulce canto'),('JuanDeArguijo','Ya el joven fuerte que con muestra hermosa'),('Quevedo','Estábase la efesia cazadora'),('Quevedo','Si con los mismos ojos que leyeres'),('Quevedo','La voluntad de Dios por grillos tienes'),('Quevedo','Escondido debajo de tu armada'),('Quevedo','Mi madre tuve en ásperas montañas'),('Quevedo','Sola en ti, Lesbia, vemos ha perdido'),('Quevedo','Llegó a los pies de Cristo Madalena')]
flores6=[]
for author,incipit in FLORES_PHASE6:
    row,method=resolve_incipit(author,incipit); ok=row is not None
    if ok:
        add_constraint([row.n_id],'not_after',1605,'B','Flores_1605_attestation_TAQ','Pedro Espinosa, Primera parte de Flores de poetas ilustres de España (1605)'); set_circulation([row.n_id],1605,'Flores de poetas ilustres attestation','Pedro Espinosa, Primera parte de Flores de poetas ilustres de España (1605)')
    flores6.append({'author':author,'incipit':incipit,'resolution':method,'n_id':None if row is None else row.n_id,'resolved':ok})
flores6=pd.DataFrame(flores6); assert int(flores6.resolved.sum())==9
primary6=temporal[temporal.chronology_status.eq('primary_dated')].copy(); assert len(primary6)==79; assert primary6.author_dir.nunique()==3

# ---- Phase 7 ----
FLORES_PHASE7=FLORES_PHASE6+[('JuanDeArguijo','La tirana codicia del hermano')]; flores7=[]
for author,incipit in FLORES_PHASE7:
    row,method=resolve_incipit(author,incipit); ok=row is not None
    if ok:
        add_constraint([row.n_id],'not_after',1603,'B','Flores_1603_dedication_approval_TAQ','Flores dedication 20 Sep 1603; approval 24 Nov 1603; publication 1605'); set_circulation([row.n_id],1605,'Flores de poetas ilustres attestation','Pedro Espinosa, Primera parte de Flores de poetas ilustres de España (1605)')
    flores7.append({'author':author,'incipit':incipit,'resolution':method,'n_id':None if row is None else row.n_id,'constraint_1603_added':ok})
flores7=pd.DataFrame(flores7); assert int(flores7.constraint_1603_added.sum())==10
HERRERA_CANDIDATES=[
 {'incipit':'Temiendo tu valor, tu ardiente espada','lo':1574,'hi':1574,'basis':'Alameda_CarlosV_event_chronology'},
 {'incipit':'Vos, celebrando al son de noble lira','lo':1578,'hi':1579,'basis':'Barahona_Granada_residence_interval'},
 {'incipit':'Esconde tardo Bágrada','lo':1573,'hi':1574,'basis':'Bazan_Tunis_campaign_interval'},
 {'incipit':'Ya que el sujeto reino Lusitano','lo':1580,'hi':1582,'basis':'Portugal_annexation_to_H_interval'},
 {'incipit':'Pongan en tu sepulcro','lo':1578,'hi':1578,'basis':'DonJuan_de_Austria_death_1578'},
]
herrera_audit=[]
for c in HERRERA_CANDIDATES:
    row,method=resolve_incipit('FernandoDeHerrera',c['incipit']); ok=row is not None
    if ok: assign_primary([row.n_id],c['lo'],c['hi'],'B',c['basis'],'López Bueno ed., Algunas obras (1998/2011 digital) + poem-specific scholarly chronology')
    herrera_audit.append({**c,'resolution':method,'n_id':None if row is None else row.n_id,'assigned':ok})
herrera_audit=pd.DataFrame(herrera_audit); primary7=temporal[temporal.chronology_status.eq('primary_dated')].copy(); assert len(primary7)==84; assert int(herrera_audit.assigned.sum())==5
phase7_ids=set(primary7.n_id)
print('BASELINES REPRODUCED: P5=76 | P6=79 | P7=84 | Herrera P7=5 | Flores1603=10/13')

BASELINES REPRODUCED: P5=76 | P6=79 | P7=84 | Herrera P7=5 | Flores1603=10/13


## Phase 8 — Transition + Baroque chronology acquisition

Phase 8 adds **no new Góngora chronology**. The objective is to reduce author concentration and fill two historical gaps: a second independently represented `Transition` author and at least one `Baroque` primary anchor.

### Pedro Espinosa
Bonilla Cerezo (2007), following López Estrada's chronology, associates four named sonnets with Espinosa's 1594–1596 *período de felicidad*. These are tested as confidence-B bounded scholarly intervals.

### Francisco de Quevedo
Only poem-specific chronological arguments are tested. Menéndez Pelayo's 1899 letter to Rodríguez Marín explicitly dates one sonnet to 1609 and places four Aminta sonnets around 1611. Three sonnets on the death of Henry IV are tested at 1610, and the Osuna memorial sonnet at 1624. These remain confidence B: historical/scholarly reconstruction, not autograph composition records.

Negative rule: a date is never assigned merely because an author, event, or publication is mentioned. The incipit must resolve uniquely in the pinned Navarro corpus and the scholarly source must provide a poem-specific chronological argument.

In [ ]:
ESPINOSA_CANDIDATES=[
 {'incipits':['Llegó diciembre sobre el cierzo helado'],'lo':1594,'hi':1596,'basis':'Espinosa_happiness_period_1594_1596'},
 {'incipits':['Levantaba, gigante en pensamiento','Levantaba gigante en pensamiento'],'lo':1594,'hi':1596,'basis':'Espinosa_happiness_period_1594_1596'},
 {'incipits':['El sol a noble furia se provoca'],'lo':1594,'hi':1596,'basis':'Espinosa_happiness_period_1594_1596'},
 {'incipits':['Pues son vuestros pinceles, Mohedano','Pues son vuestros pinceles Mohedano'],'lo':1594,'hi':1596,'basis':'Espinosa_happiness_period_1594_1596'},
]
espinosa_audit=[]
ESP_SOURCE='Rafael Bonilla Cerezo, Góngora entre azahares: la Epístola I a Heliodoro de Pedro Espinosa, Analecta Malacitana XXX.1 (2007), note 4, following F. López Estrada, Poesías completas (1975), pp. 4–8'
for c in ESPINOSA_CANDIDATES:
    row,method,matched_variant=resolve_any_incipit('PedroEspinosa',c['incipits']); ok=row is not None
    if ok: assign_primary([row.n_id],c['lo'],c['hi'],'B',c['basis'],ESP_SOURCE)
    espinosa_audit.append({'incipit_query':' | '.join(c['incipits']),'resolution':method,'matched_variant':matched_variant,'n_id':None if row is None else row.n_id,'matched_first_line':None if row is None else row.first_line,'lo':c['lo'],'hi':c['hi'],'assigned':ok})
espinosa_audit=pd.DataFrame(espinosa_audit)
print('PHASE 8A — Espinosa 1594–1596 audit'); display(espinosa_audit)

PHASE 8A — Espinosa 1594–1596 audit


,incipit_query,resolution,matched_variant,n_id,matched_first_line,lo,hi,assigned
0,Llegó diciembre sobre el cierzo helado,exact_first_line,Llegó diciembre sobre el cierzo helado,PedroEspinosa::PedroEspinosa_2.xml,"Llegó diciembre sobre el cierzo helado,",1594,1596,True
1,"Levantaba, gigante en pensamiento | Levantaba ...",exact_first_line,"Levantaba, gigante en pensamiento",PedroEspinosa::PedroEspinosa_19.xml,Levantaba (gigante en pensamiento),1594,1596,True
2,El sol a noble furia se provoca,exact_first_line,El sol a noble furia se provoca,PedroEspinosa::PedroEspinosa_4.xml,El sol a noble furia se provoca,1594,1596,True
3,"Pues son vuestros pinceles, Mohedano | Pues so...",exact_first_line,"Pues son vuestros pinceles, Mohedano",PedroEspinosa::PedroEspinosa_5.xml,"Pues son vuestros pinceles, Mohedano,",1594,1596,True


In [ ]:
QUEVEDO_CANDIDATES=[
 {'incipits':['Así, sagrado mar, nunca te oprima','Ansí, sagrado mar, nunca te oprima'],'lo':1609,'hi':1609,'basis':'MenendezPelayo_letter_Carrillo_sonnet_1609','source':'Menéndez Pelayo to Francisco Rodríguez Marín, 6 Jul 1899, Epistolario XV, carta 386'},
 {'incipits':['Aminta, si a tu pecho y a tu cuello'],'lo':1611,'hi':1611,'basis':'MenendezPelayo_Aminta_group_1611','source':'Menéndez Pelayo to Francisco Rodríguez Marín, 6 Jul 1899, Epistolario XV, carta 386'},
 {'incipits':['Lo que me quita en fuego, me da en nieve'],'lo':1611,'hi':1611,'basis':'MenendezPelayo_Aminta_group_1611','source':'Menéndez Pelayo to Francisco Rodríguez Marín, 6 Jul 1899, Epistolario XV, carta 386'},
 {'incipits':['Aminta, para mí cualquiera día'],'lo':1611,'hi':1611,'basis':'MenendezPelayo_Aminta_group_1611','source':'Menéndez Pelayo to Francisco Rodríguez Marín, 6 Jul 1899, Epistolario XV, carta 386'},
 {'incipits':['Ver relucir en llamas encendido'],'lo':1611,'hi':1611,'basis':'MenendezPelayo_Aminta_group_1611','source':'Menéndez Pelayo to Francisco Rodríguez Marín, 6 Jul 1899, Epistolario XV, carta 386'},
 {'incipits':['Su mano coronó su cuello ardiente'],'lo':1610,'hi':1610,'basis':'HenryIV_death_memorial_1610','source':'BVMC, Sonetos de Quevedo: Inscripción al túmulo del Rey de Francia Enrique IV'},
 {'incipits':['No pudo haber estrella que infamase'],'lo':1610,'hi':1610,'basis':'HenryIV_death_memorial_1610','source':'BVMC, Sonetos de Quevedo: muerte del mismo rey Enrique IV'},
 {'incipits':['No llegó a tanta envidia de los hados'],'lo':1610,'hi':1610,'basis':'HenryIV_death_memorial_1610','source':'BVMC, Sonetos de Quevedo: muerte del Cuarto Enrico, Rey de Francia'},
 {'incipits':['Faltar pudo su patria al grande Osuna'],'lo':1624,'hi':1624,'basis':'Osuna_death_memorial_1624','source':'CVC/BVMC Quevedo chronology and commentary: memorial sonnet to Pedro Girón, Duke of Osuna, died 1624'},
]
quevedo_audit=[]
for c in QUEVEDO_CANDIDATES:
    row,method,matched_variant=resolve_any_incipit('Quevedo',c['incipits']); ok=row is not None
    if ok: assign_primary([row.n_id],c['lo'],c['hi'],'B',c['basis'],c['source'])
    quevedo_audit.append({'incipit_query':' | '.join(c['incipits']),'resolution':method,'matched_variant':matched_variant,'n_id':None if row is None else row.n_id,'matched_first_line':None if row is None else row.first_line,'lo':c['lo'],'hi':c['hi'],'assigned':ok,'basis':c['basis']})
quevedo_audit=pd.DataFrame(quevedo_audit)
print('PHASE 8B — Quevedo poem-specific chronology audit'); display(quevedo_audit)

PHASE 8B — Quevedo poem-specific chronology audit


,incipit_query,resolution,matched_variant,n_id,matched_first_line,lo,hi,assigned,basis
0,"Así, sagrado mar, nunca te oprima | Ansí, sagr...",exact_first_line,"Ansí, sagrado mar, nunca te oprima",Quevedo::Quevedo_131.xml,"Ansí, sagrado mar, nunca te oprima",1609,1609,True,MenendezPelayo_letter_Carrillo_sonnet_1609
1,"Aminta, si a tu pecho y a tu cuello",exact_first_line,"Aminta, si a tu pecho y a tu cuello",Quevedo::Quevedo_69.xml,"Aminta, si a tu pecho y a tu cuello",1611,1611,True,MenendezPelayo_Aminta_group_1611
2,"Lo que me quita en fuego, me da en nieve",exact_first_line,"Lo que me quita en fuego, me da en nieve",Quevedo::Quevedo_70.xml,"Lo que me quita en fuego, me da en nieve",1611,1611,True,MenendezPelayo_Aminta_group_1611
3,"Aminta, para mí cualquiera día",exact_first_line,"Aminta, para mí cualquiera día",Quevedo::Quevedo_72.xml,"Aminta, para mí cualquiera día",1611,1611,True,MenendezPelayo_Aminta_group_1611
4,Ver relucir en llamas encendido,exact_first_line,Ver relucir en llamas encendido,Quevedo::Quevedo_76.xml,"Ver relucir, en llamas encendido,",1611,1611,True,MenendezPelayo_Aminta_group_1611
5,Su mano coronó su cuello ardiente,exact_first_line,Su mano coronó su cuello ardiente,Quevedo::Quevedo_42.xml,Su mano coronó su cuello ardiente,1610,1610,True,HenryIV_death_memorial_1610
6,No pudo haber estrella que infamase,exact_first_line,No pudo haber estrella que infamase,Quevedo::Quevedo_43.xml,No pudo haber estrella que infamase,1610,1610,True,HenryIV_death_memorial_1610
7,No llegó a tanta envidia de los hados,exact_first_line,No llegó a tanta envidia de los hados,Quevedo::Quevedo_45.xml,"No llegó a tanta envidia de los hados,",1610,1610,True,HenryIV_death_memorial_1610
8,Faltar pudo su patria al grande Osuna,exact_first_line,Faltar pudo su patria al grande Osuna,Quevedo::Quevedo_44.xml,"Faltar pudo su patria al grande Osuna,",1624,1624,True,Osuna_death_memorial_1624


In [ ]:
# External literary-historical classification: validation only
hist_rows=[('GarcilasoDeLaVega','Innovation'),('JuanBoscan','Innovation'),('FernandoDeHerrera','Renovation'),('JuanDeArguijo','Transition'),('JuanDeJauregui','Transition'),('Cervantes','Transition'),('PedroEspinosa','Transition'),('LuisCarrilloySotomayor','Transition'),('Gongora','Culmination'),('LopeDeVega_1','Culmination'),('LopeDeVega_2','Culmination'),('Quevedo','Baroque')]
hist=pd.DataFrame(hist_rows,columns=['author_dir','lopez_bueno_2006']); stage_order={'Innovation':0,'Renovation':1,'Transition':2,'Culmination':3,'Baroque':4}; hist['hist_stage_order']=hist.lopez_bueno_2006.map(stage_order).astype('Int64'); hist['auxiliary_role']='external_validation_only'; priority_A=set(hist.author_dir); temporal=temporal.merge(hist,on='author_dir',how='left')

def identifiability_class(r):
    if r.chronology_status=='primary_dated': return 'primary_interval'
    if pd.notna(r.composition_not_before) and pd.notna(r.composition_not_after): return 'bounded_constraint'
    if pd.notna(r.composition_not_before) or pd.notna(r.composition_not_after): return 'one_sided_constraint'
    if pd.notna(r.circulation_year): return 'circulation_only'
    if r.sensitivity_status=='sensitivity_only': return 'sensitivity_only'
    return 'unconstrained'
temporal['identifiability_class']=temporal.apply(identifiability_class,axis=1)
primary=temporal[temporal.chronology_status.eq('primary_dated')].copy(); primary8_new=primary[~primary.n_id.isin(phase7_ids)].copy()
# Hard integrity: primary intervals cannot contradict tighter one-sided bounds.
bad=[]
for r in primary.itertuples(index=False):
    if pd.notna(r.composition_not_before) and int(r.composition_max)<int(r.composition_not_before): bad.append((r.n_id,'not_before'))
    if pd.notna(r.composition_not_after) and int(r.composition_min)>int(r.composition_not_after): bad.append((r.n_id,'not_after'))
assert not bad,bad[:10]
pc=primary.groupby('author_dir').size().sort_values(ascending=False); p=pc/pc.sum(); effective_authors=float(math.exp(-(p*p.map(math.log)).sum())); top_author_share=float(p.iloc[0]); gongora_share=float(p.get('Gongora',0.0)); author_count=int(primary.author_dir.nunique())
primary_hist=primary.merge(hist[['author_dir','lopez_bueno_2006','hist_stage_order']],on='author_dir',how='left',suffixes=('','_hist')); stage_count=int(primary_hist.lopez_bueno_2006_hist.nunique()); transition_authors=int(primary_hist.loc[primary_hist.lopez_bueno_2006_hist.eq('Transition'),'author_dir'].nunique()); baroque_primary=int(primary_hist.lopez_bueno_2006_hist.eq('Baroque').sum())
stage_rows=[]
for stage,order in stage_order.items():
    ids=set(hist.loc[hist.lopez_bueno_2006.eq(stage),'author_dir']); z=temporal[temporal.author_dir.isin(ids)]; zp=z[z.chronology_status.eq('primary_dated')]
    stage_rows.append({'stage':stage,'order':order,'total_poems':len(z),'primary_poems':len(zp),'primary_authors':zp.author_dir.nunique()})
stage_summary=pd.DataFrame(stage_rows).sort_values('order')
metrics=pd.DataFrame([{'primary_poems':len(primary),'phase8_new_primary':len(primary8_new),'primary_author_groups':author_count,'primary_historiographic_stages':stage_count,'transition_primary_authors':transition_authors,'baroque_primary_poems':baroque_primary,'effective_primary_authors':round(effective_authors,3),'top_author_share':round(top_author_share,3),'gongora_primary_share':round(gongora_share,3),'espinosa_phase8_assigned':int(espinosa_audit.assigned.sum()),'quevedo_phase8_assigned':int(quevedo_audit.assigned.sum())}])
guardrails=pd.DataFrame([
 {'diagnostic':'primary_author_groups >= 5','value':author_count,'pass':author_count>=5},
 {'diagnostic':'all 5 historiographic stages represented','value':stage_count,'pass':stage_count==5},
 {'diagnostic':'Transition primary authors >= 2','value':transition_authors,'pass':transition_authors>=2},
 {'diagnostic':'Baroque primary poems >= 1','value':baroque_primary,'pass':baroque_primary>=1},
 {'diagnostic':'effective_primary_authors >= 3','value':round(effective_authors,3),'pass':effective_authors>=3},
 {'diagnostic':'top_author_share < 0.65','value':round(top_author_share,3),'pass':top_author_share<0.65},
]); ready=bool(guardrails['pass'].all())
source_gap_worklist=pd.DataFrame([
 {'author':'JuanDeArguijo','status':'one-sided evidence only','next_action':'seek poem-specific lower bounds or dated manuscript/redaction evidence; do not promote 1599–1605 source-level statements wholesale'},
 {'author':'JuanDeJauregui','status':'known dated poems may fall outside Navarro 23-sonnet subset','next_action':'verify text identity before any chronology transfer'},
 {'author':'LuisCarrilloySotomayor','status':'1609 Así/Ansí sagrado mar is a Quevedo sonnet addressed to Carrillo, not a Carrillo poem','next_action':'do not use this evidence for Carrillo chronology'},
])
print('PHASE 8 IDENTIFIABILITY SUMMARY'); display(metrics); print('\nPrimary counts by author'); display(pc.rename('primary_poems').to_frame()); print('\nHistoriographic-stage coverage'); display(stage_summary); print('\nOperational readiness guardrails'); display(guardrails); print('\nPhase-8 source-gap worklist'); display(source_gap_worklist)

PHASE 8 IDENTIFIABILITY SUMMARY


,primary_poems,phase8_new_primary,primary_author_groups,primary_historiographic_stages,transition_primary_authors,baroque_primary_poems,effective_primary_authors,top_author_share,gongora_primary_share,espinosa_phase8_assigned,quevedo_phase8_assigned
0,97,13,6,5,2,9,3.43,0.598,0.598,4,9



Primary counts by author


,primary_poems
author_dir,
Gongora,58
GarcilasoDeLaVega,18
Quevedo,9
FernandoDeHerrera,5
PedroEspinosa,4
Cervantes,3



Historiographic-stage coverage


,stage,order,total_poems,primary_poems,primary_authors
0,Innovation,0,138,18,1
1,Renovation,1,320,5,1
2,Transition,2,240,7,2
3,Culmination,3,1461,58,1
4,Baroque,4,517,9,1



Operational readiness guardrails


,diagnostic,value,pass
0,primary_author_groups >= 5,6.000,True
1,all 5 historiographic stages represented,5.000,True
2,Transition primary authors >= 2,2.000,True
3,Baroque primary poems >= 1,9.000,True
4,effective_primary_authors >= 3,3.430,True
5,top_author_share < 0.65,0.598,True



Phase-8 source-gap worklist


,author,status,next_action
0,JuanDeArguijo,one-sided evidence only,seek poem-specific lower bounds or dated manus...
1,JuanDeJauregui,known dated poems may fall outside Navarro 23-...,verify text identity before any chronology tra...
2,LuisCarrilloySotomayor,1609 Así/Ansí sagrado mar is a Quevedo sonnet ...,do not use this evidence for Carrillo chronology


In [ ]:
OUT=Path('/content/gasr_phase8_outputs'); OUT.mkdir(exist_ok=True)
constraints=pd.DataFrame(constraint_log)
temporal.to_csv(OUT/'temporal_master_phase8.csv',index=False); constraints.to_csv(OUT/'phase8_constraint_evidence.csv',index=False); espinosa_audit.to_csv(OUT/'phase8_espinosa_anchor_audit.csv',index=False); quevedo_audit.to_csv(OUT/'phase8_quevedo_anchor_audit.csv',index=False); metrics.to_csv(OUT/'phase8_identifiability_metrics.csv',index=False); stage_summary.to_csv(OUT/'phase8_historiographic_stage_coverage.csv',index=False); guardrails.to_csv(OUT/'phase8_readiness_guardrails.csv',index=False); source_gap_worklist.to_csv(OUT/'phase8_source_gap_worklist.csv',index=False)
assert len(primary7)==84; assert int(herrera_audit.assigned.sum())==5; assert int(flores7.constraint_1603_added.sum())==10; assert int((temporal.sensitivity_status=='sensitivity_only').sum())==100
print('\nPHASE 8 CHECKPOINT'); print('------------------'); print('Phase-7 baseline preserved: primary=84.'); print('Espinosa primary additions:',int(espinosa_audit.assigned.sum()),'/',len(espinosa_audit)); print('Quevedo primary additions:',int(quevedo_audit.assigned.sum()),'/',len(quevedo_audit)); print('Phase-8 primary total:',len(primary)); print('Primary author groups:',author_count); print('Primary historiographic stages:',stage_count,'/ 5'); print('Transition primary authors:',transition_authors); print('Baroque primary poems:',baroque_primary); print('Effective primary authors:',round(effective_authors,3)); print('Top-author share:',round(top_author_share,3)); print('Semantic-timeline readiness:','READY' if ready else 'NOT READY'); print('Outputs:',OUT)


PHASE 8 CHECKPOINT
------------------
Phase-7 baseline preserved: primary=84.
Espinosa primary additions: 4 / 4
Quevedo primary additions: 9 / 9
Phase-8 primary total: 97
Primary author groups: 6
Primary historiographic stages: 5 / 5
Transition primary authors: 2
Baroque primary poems: 9
Effective primary authors: 3.43
Top-author share: 0.598
Semantic-timeline readiness: READY
Outputs: /content/gasr_phase8_outputs
